In [ ]:
import os

print("⏳ Installing WhisperX and dependencies...")

!pip uninstall -y torch torchvision torchaudio transformers pyannote.audio whisperx

!pip install -q torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu118


!pip install -q pyannote.audio==3.1.1


!pip install -q git+https://github.com/m-bain/whisperx.git

!pip install -q gradio pandas

!pip install -q "transformers>=4.41.0" "numpy<2.0.0"

print("✅ Installation complete. Restarting kernel...")

os._exit(0)

⏳ Installing WhisperX and dependencies...
Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
ERROR: Could not find a version that satisfies the requirement torch==2.1.2 (from versions: 2.2.0+cu118, 2.2.1+cu118, 2.2.2+cu118, 2.3.0+cu118, 2.3.1+cu118, 2.4.0+cu118, 2.4.1+cu118, 2.5.0+cu118, 2.5.1+cu118, 2.6.0+cu118, 2.7.0+cu118, 2.7.1+cu118)
ERROR: No matching distribution found for torch==2.1.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:

import whisperx
import torch
import torch.serialization
import gradio as gr
import pandas as pd
import gc
import os
import functools

# START PATCH for PyTorch > 2.6 compatibility
if hasattr(torch.serialization, "load"):
    if torch.load != torch.serialization.load:
        print("🔄 Resetting torch.load to avoid recursion...")
        torch.load = torch.serialization.load

_original_torch_load = torch.load
@functools.wraps(_original_torch_load)
def patched_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)

torch.load = patched_torch_load
print("✅ Patched torch.load to set weights_only=False for legacy model support")
# END PATCH

print("✅ Torch version:", torch.__version__)
print("✅ CUDA available:", torch.cuda.is_available())

# ---------- PHASE 3: CONFIG ----------
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")  # Colab Secret named HF_TOKEN
if not HF_TOKEN:
    raise ValueError("HF_TOKEN secret not found in Colab secrets")

login(token=HF_TOKEN)

device = "cuda"
batch_size = 4
compute_type = "int8"

# ---------- PHASE 4: LOAD MODELS ----------
print("⏳ Loading Whisper Large-V3...")
model = whisperx.load_model(
    "large-v3",
    device,
    compute_type=compute_type
)

print("⏳ Loading Alignment Model...")
model_a, metadata = whisperx.load_align_model(
    language_code="en",
    device=device
)

print("⏳ Loading Diarization Model...")
from whisperx.diarize import DiarizationPipeline

# New WhisperX API: do NOT pass use_auth_token
try:
    diarize_model = DiarizationPipeline(device=device)
    print("✅ Diarization Pipeline loaded")
except TypeError:
    # Fallback for alternate builds
    diarize_model = DiarizationPipeline(token=HF_TOKEN, device=device)
    print("✅ Diarization Pipeline loaded with token")

print("✅ All models loaded successfully!")

# ---------- PHASE 5: DIARIZATION FORMAT FIX ----------
def format_diarization(diarize_segments):
    if isinstance(diarize_segments, pd.DataFrame):
        return diarize_segments

    records = []
    for turn, _, speaker in diarize_segments.itertracks(yield_label=True):
        records.append({
            "start": turn.start,
            "end": turn.end,
            "speaker": speaker
        })
    return pd.DataFrame(records)

# ---------- PHASE 6: PROCESSING FUNCTION ----------
def process_audio(audio_path, num_speakers=None, min_speakers=None, max_speakers=None):
    if not audio_path:
        return [{"error": "No file uploaded"}]

    try:
        print(f"🎤 Processing: {os.path.basename(audio_path)}")

        audio = whisperx.load_audio(audio_path)

        # A. Transcribe
        print("   ... Transcribing")
        result = model.transcribe(audio, batch_size=batch_size)

        # B. Align
        print("   ... Aligning")
        result = whisperx.align(
            result["segments"],
            model_a,
            metadata,
            audio,
            device,
            return_char_alignments=False
        )

        # Handle optional speaker counts
        if num_speakers is not None and int(num_speakers) > 0:
            min_speakers = int(num_speakers)
            max_speakers = int(num_speakers)

        if min_speakers is not None: min_speakers = int(min_speakers)
        if max_speakers is not None: max_speakers = int(max_speakers)

        print(f"👥 Diarizing with min_speakers={min_speakers}, max_speakers={max_speakers}")

        # C. Diarize
        diarize_segments = diarize_model(audio, min_speakers=min_speakers, max_speakers=max_speakers)

        # Debug: Print Unique Speakers Found (Using Pandas)
        if hasattr(diarize_segments, "itertracks"):
             # It's a pyannote Annotation object
             print(f"   ---> Raw Annotation: {diarize_segments}")

        # try removing this once
        diarize_df = format_diarization(diarize_segments)

        unique_speakers = diarize_df['speaker'].unique() if not diarize_df.empty else []
        print(f"   ---> Unique Speakers Found: {unique_speakers}")


        # D. Merge speakers
        print("   ... Merging")
        final_result = whisperx.assign_word_speakers(
            diarize_df,
            result
        )

        # E. Clean output
        output = []
        for seg in final_result["segments"]:
            output.append({
                "speaker": seg.get("speaker", "Unknown"),
                "start": round(seg["start"], 2),
                "end": round(seg["end"], 2),
                "text": seg["text"].strip()
            })

        return output

    except Exception as e:
        print(f"❌ Error: {str(e)}")
        return [{"error": str(e)}]

    finally:
        gc.collect()
        torch.cuda.empty_cache()

🔄 Resetting torch.load to avoid recursion...
✅ Patched torch.load to set weights_only=False for legacy model support
✅ Torch version: 2.8.0+cu128
✅ CUDA available: True
⏳ Loading Whisper Large-V3...
2026-05-23 13:19:30 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-05-23 13:19:30 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.4. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.4. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`


⏳ Loading Alignment Model...
⏳ Loading Diarization Model...
2026-05-23 13:19:32 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-community-1


config.yaml:   0%|          | 0.00/444 [00:00<?, ?B/s]

segmentation/pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

plda/xvec_transform.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

plda/plda.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

embedding/pytorch_model.bin:   0%|          | 0.00/26.6M [00:00<?, ?B/s]

✅ Diarization Pipeline loaded
✅ All models loaded successfully!


In [4]:

app = gr.Interface(
    fn=process_audio,
    inputs=[
        gr.Audio(type="filepath", label="Upload Audio"),
        gr.Number(label="Number of Speakers (Optional: Exact)", value=None, precision=0),
        gr.Number(label="Min Speakers (Optional)", value=None, precision=0),
        gr.Number(label="Max Speakers (Optional)", value=None, precision=0)
    ],
    outputs=gr.JSON(),
    title="🎙️ WhisperX Speaker Diarization API",
    description="Upload audio → transcription + timestamps + speakers. Provide speaker counts for better results."
)

app.queue().launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8313d5f16e9619a551.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🎤 Processing: blob
   ... Transcribing
2026-05-23 13:41:23 - whisperx.asr - WARNING - Audio is shorter than 30s, language detection may be inaccurate
2026-05-23 13:41:23 - whisperx.asr - INFO - Detected language: en (1.00) in first 30s of audio
   ... Aligning
👥 Diarizing with min_speakers=2, max_speakers=2
   ---> Unique Speakers Found: ['SPEAKER_00' 'SPEAKER_01']
   ... Merging
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://8313d5f16e9619a551.gradio.live
